<a href="https://colab.research.google.com/github/fitraadhim/244107020089_MachineLearning/blob/main/JS06ANN/prak3JS06ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679800 sha256=dcf6e56f376ec60bbe5b490e5c46d2796c951012448559eb4cc0ecc89ff87ff3
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib


In [2]:
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors

# ===========================
# 1. Buat data 2D acak
# ===========================
num_elements = 1000
dim = 2
data = np.random.random((num_elements, dim)).astype(np.float32)

# Query point
query = np.array([[0.5, 0.5]], dtype=np.float32)
k = 5  # cari 5 tetangga terdekat

# ===========================
# 2. Exact NN (Brute Force)
# ===========================
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

start = time.time()
distances, indices = nn.kneighbors(query)
end = time.time()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Waktu:", end - start, "detik")

# ===========================
# 3. HNSW
# ===========================
# Inisialisasi index HNSW
p = hnswlib.Index(space='l2', dim=dim)

# Ukuran maksimum elemen yang bisa ditampung
p.init_index(max_elements=num_elements, ef_construction=100, M=16)

# Tambahkan data
p.add_items(data)

# Set parameter pencarian
p.set_ef(50)   # tradeoff speed vs accuracy

start = time.time()
labels, distances = p.knn_query(query, k=k)
end = time.time()

print("\n=== HNSW ===")
print("Indices:", labels)
print("Distances:", distances)
print("Waktu:", end - start, "detik")


=== Exact NN ===
Indices: [[935 436 999 985 704]]
Distances: [[0.03717297 0.04263567 0.04462878 0.04952153 0.05089222]]
Waktu: 0.03556966781616211 detik

=== HNSW ===
Indices: [[935 436 999 985 704]]
Distances: [[0.00138183 0.0018178  0.00199173 0.00245238 0.00259002]]
Waktu: 0.000156402587890625 detik


In [3]:
def run_hnsw_experiment(dim, num_elements, space_metric):
    # 1. Buat data acak dan query point
    np.path = 42
    data = np.random.random((num_elements, dim)).astype(np.float32)
    query = np.array([[0.5] * dim], dtype=np.float32)
    k = 5

    # 2. Exact NN (Brute Force)
    metric_sklearn = 'euclidean' if space_metric == 'l2' else 'cosine'
    nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric=metric_sklearn)
    nn.fit(data)

    start = time.time()
    distances_exact, indices_exact = nn.kneighbors(query)
    time_exact = time.time() - start

    # 3. HNSW
    p = hnswlib.Index(space=space_metric, dim=dim)
    p.init_index(max_elements=num_elements, ef_construction=100, M=16)
    p.add_items(data)
    p.set_ef(50)

    start = time.time()
    labels_hnsw, distances_hnsw = p.knn_query(query, k=k)
    time_hnsw = time.time() - start

    # Print Hasil
    print(f"--- Metric: {space_metric.upper()} | Dim: {dim}D | Data: {num_elements} ---")
    print("Exact Indices:", indices_exact)
    print("HNSW Indices:", labels_hnsw)
    print("Waktu Exact:", round(time_exact, 6), "s | Waktu HNSW:", round(time_hnsw, 6), "s\n")

# Jalankan eksperimen untuk L2 dan Cosine (1.000 data dan 1.000.000 data)
run_hnsw_experiment(dim=2, num_elements=1000, space_metric='l2')
run_hnsw_experiment(dim=5, num_elements=1000, space_metric='l2')
run_hnsw_experiment(dim=2, num_elements=1000000, space_metric='l2')
run_hnsw_experiment(dim=5, num_elements=1000000, space_metric='l2')

run_hnsw_experiment(dim=2, num_elements=1000, space_metric='cosine')
run_hnsw_experiment(dim=5, num_elements=1000, space_metric='cosine')
run_hnsw_experiment(dim=2, num_elements=1000000, space_metric='cosine')
run_hnsw_experiment(dim=5, num_elements=1000000, space_metric='cosine')

--- Metric: L2 | Dim: 2D | Data: 1000 ---
Exact Indices: [[125 216 806 703 550]]
HNSW Indices: [[125 216 806 703 550]]
Waktu Exact: 0.001081 s | Waktu HNSW: 6.3e-05 s

--- Metric: L2 | Dim: 5D | Data: 1000 ---
Exact Indices: [[570 569 806 730 194]]
HNSW Indices: [[570 569 806 730 194]]
Waktu Exact: 0.000617 s | Waktu HNSW: 7.5e-05 s

--- Metric: L2 | Dim: 2D | Data: 1000000 ---
Exact Indices: [[656345 813404 611279 192032 697643]]
HNSW Indices: [[656345 813404 611279 192032 697643]]
Waktu Exact: 0.031554 s | Waktu HNSW: 0.000101 s

--- Metric: L2 | Dim: 5D | Data: 1000000 ---
Exact Indices: [[368276 374624 774327 680557 159359]]
HNSW Indices: [[368276 374624 774327 680557 159359]]
Waktu Exact: 0.029783 s | Waktu HNSW: 0.000184 s

--- Metric: COSINE | Dim: 2D | Data: 1000 ---
Exact Indices: [[963 536 161 818 452]]
HNSW Indices: [[963 536 161 818 452]]
Waktu Exact: 0.015793 s | Waktu HNSW: 5.6e-05 s

--- Metric: COSINE | Dim: 5D | Data: 1000 ---
Exact Indices: [[773 883  20 589 941]]
HNS